In [1]:
def init_ee(project_id=None):
    """
    Authenticates and initializes Google Earth Engine.
    """
    print("Checking Earth Engine session...")
    try:
        import ee, sys
        ee.Initialize(project=project_id) if project_id else ee.Initialize()
        print("Earth Engine is already initialized.")
    except Exception as _:
        print("No active session. Starting authentication flow...")
        import ee
        ee.Authenticate()
        ee.Initialize(project=project_id) if project_id else ee.Initialize()
        print("Earth Engine initialized after authentication.")
    finally:
        import sys
        pv = sys.version.split()[0]
        print(f"Python {pv} | EE project: {project_id if project_id else 'default'}")

EE_PROJECT_ID = "replicating-paper"

init_ee(EE_PROJECT_ID)


Checking Earth Engine session...
Earth Engine is already initialized.
Python 3.12.12 | EE project: replicating-paper


In [2]:

SELECT_REGION = "sanjay_van"
# SELECT_REGION = "sundarbans"
# SELECT_REGION = "western_ghats"
# SELECT_REGION = "himalayas"

# All four regions — rectangles [minLon, minLat, maxLon, maxLat]
REGIONS = {
    "sanjay_van": {
        "name": "Sanjay Van (Delhi) — urban dry-deciduous / scrub woodland",
        "bbox": [77.17, 28.52, 77.18, 28.54],
    },
    "sundarbans": {
        "name": "Sundarbans — coastal mangrove (tidal)",
        "bbox": [88.85, 21.85, 88.87, 21.87],
    },
    "western_ghats": {
        "name": "Western Ghats — tropical evergreen / montane",
        "bbox": [75.74, 11.27, 75.76, 11.29],
    },
    "himalayas": {
        "name": "Himalayas — sub-alpine / alpine (rugged)",
        "bbox": [78.40, 33.20, 78.42, 33.22],
    }
}

# Time & grid
YEAR = 2025
CRS  = "EPSG:32643"
SCALE = 10
SAMPLE_SCALE = 50

WET_MONTHS = [7, 8, 9]                         # Jul–Sep
DRY_MONTHS = [1, 2, 3, 4, 10, 11, 12]          # Jan–Apr & Oct–Dec

PHENO_COEFFS_ASSET = "users/drewhart/Terasaki_Hart_2024_LSP_diversity_asynchrony_main_map_results"  # from paper
HEIGHT_ASSET        = "users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1"                              # ETH 10 m height (2020)

print("Config OK:")
print(" SELECT_REGION =", SELECT_REGION, " -> ", REGIONS[SELECT_REGION]["name"])
print(" YEAR =", YEAR, "| CRS =", CRS, "| SCALE =", SCALE, "| SAMPLE_SCALE =", SAMPLE_SCALE)
print(" WET_MONTHS =", WET_MONTHS, "| DRY_MONTHS =", DRY_MONTHS)
print(" PHENO_COEFFS_ASSET =", PHENO_COEFFS_ASSET)
print(" HEIGHT_ASSET        =", HEIGHT_ASSET)


Config OK:
 SELECT_REGION = sanjay_van  ->  Sanjay Van (Delhi) — urban dry-deciduous / scrub woodland
 YEAR = 2025 | CRS = EPSG:32643 | SCALE = 10 | SAMPLE_SCALE = 50
 WET_MONTHS = [7, 8, 9] | DRY_MONTHS = [1, 2, 3, 4, 10, 11, 12]
 PHENO_COEFFS_ASSET = users/drewhart/Terasaki_Hart_2024_LSP_diversity_asynchrony_main_map_results
 HEIGHT_ASSET        = users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1


In [3]:
import ee

bbox = REGIONS[SELECT_REGION]["bbox"]
aoi = ee.Geometry.Rectangle(bbox, proj=None, geodesic=False)
print("AOI name:", REGIONS[SELECT_REGION]["name"])
print("AOI bbox:", bbox)
print("AOI area (ha) ~", round(aoi.area(1).getInfo() / 1e4, 2))


AOI name: Sanjay Van (Delhi) — urban dry-deciduous / scrub woodland
AOI bbox: [77.17, 28.52, 77.18, 28.54]
AOI area (ha) ~ 217.26


In [4]:

DW       = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
S2       = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S1       = ee.ImageCollection("COPERNICUS/S1_GRD")
CHIRPS   = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
MCD64    = ee.ImageCollection("MODIS/061/MCD64A1")
VIIRSNTL = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")

H_ASSET_ID = "users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1"
try:
    H_IMAGE = ee.Image(H_ASSET_ID)
    print("✓ Height asset loaded:", H_ASSET_ID, "bands:", H_IMAGE.bandNames().getInfo())
except Exception as e:
    print("✗ Height asset failed:", H_ASSET_ID, "\n  →", e)
    raise

# --- Phenology (try known candidates; else skip for now) ---
PHENO = None
PHENO_ID = None
PHENO_CANDIDATES = [
    "users/drewhart/Terasaki_Hart_2024_LSP_diversity_asynchrony_main_map_results",
    "projects/lyrical-ring-231401/assets/Terasaki_Hart_2024_LSP_main_map_results",
]

for pid in PHENO_CANDIDATES:
    try:
        img = ee.Image(pid)
        _ = img.projection().crs().getInfo()  # force a server hit
        PHENO = img
        PHENO_ID = pid
        print("✓ Phenology asset loaded:", pid)
        print("  bands (first 15):", img.bandNames().getInfo()[:15])
        break
    except Exception as e:
        print("… not accessible:", pid)

if PHENO is None:
    print("⚠ No public phenology asset accessible from this account.")
    print("   → We will SKIP phenology features for now (everything else will run).")
    print("   → Upload *_coeffs.tif from the paper’s Zenodo archive and set PHENO_ID to your new asset; see the next cell for steps.")


✓ Height asset loaded: users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1 bands: ['b1']
… not accessible: users/drewhart/Terasaki_Hart_2024_LSP_diversity_asynchrony_main_map_results
… not accessible: projects/lyrical-ring-231401/assets/Terasaki_Hart_2024_LSP_main_map_results
⚠ No public phenology asset accessible from this account.
   → We will SKIP phenology features for now (everything else will run).
   → Upload *_coeffs.tif from the paper’s Zenodo archive and set PHENO_ID to your new asset; see the next cell for steps.


In [5]:
def _masked_const_band(name, value=0):
    return ee.Image.constant(value).rename(name).updateMask(ee.Image(0))

def _masked_s1_two_band():
    return (ee.Image.constant([0, 0])
            .rename(["VV","VH"])
            .updateMask(ee.Image(0)))

def s2_month_ndvi(year, month, region):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")
    coll  = (S2.filterBounds(region).filterDate(start, end)
               .map(mask_s2).map(add_ndvi).select("NDVI"))
    # If empty → masked placeholder band with the right name
    band_name = f"ndvi_{year}_{month:02d}"
    img = ee.Image(ee.Algorithms.If(
        coll.size().gt(0),
        coll.median().rename(band_name).clip(region),
        _masked_const_band(band_name)
    ))
    return ee.Image(img)

def s1_month(year, month, region, buffer_m=2000):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")
    # widen query to catch nearby swaths; clip later
    roi = region.buffer(buffer_m).bounds()

    coll = (S1.filterBounds(roi)
              .filterDate(start, end)
              .filter(ee.Filter.eq("instrumentMode", "IW"))
              # keep dual-pol months only (we need both VV & VH for the diff)
              .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
              .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
              .select(["VV","VH"]))

    img = ee.Image(ee.Algorithms.If(
        coll.size().gt(0),
        coll.median().clip(region),
        _masked_s1_two_band()
    ))
    return img

def s1_seasonal(year, months, region):
    ims = ee.ImageCollection.fromImages([ s1_month(year, m, region) for m in months ])
    meanVV = ims.select("VV").mean()
    meanVH = ims.select("VH").mean()
    diff   = meanVV.subtract(meanVH).rename(f"s1_vv_vh_diff_{year}")
    return meanVH.rename(f"s1_vh_mean_{year}").addBands(diff)

def s1_vh_trend(start_year, end_year, region, buffer_m=2000):

    imgs = []
    base = ee.Date.fromYMD(start_year, 1, 1)

    for y in range(start_year, end_year + 1):
        for m in range(1, 12 + 1):
            # monthly S1 (robust helper you already have)
            s1 = s1_month(y, m, region, buffer_m=buffer_m)
            vh = s1.select('VH')  # may be fully masked for some months; that's fine

            # time in years since base
            t_years = ee.Number(ee.Date.fromYMD(y, m, 15).difference(base, 'year'))

            # Make 'constant' and 't' by algebra on VH so their pixel types match VH everywhere
            ones = vh.multiply(0).add(1).rename('constant').toFloat()
            tband = vh.multiply(0).add(t_years).rename('t').toFloat()

            # Image has bands: VH (response), constant & t (predictors)
            im = vh.addBands([ones, tband]) \
                   .set('system:time_start', ee.Date.fromYMD(y, m, 15).millis())
            imgs.append(im)

    col = ee.ImageCollection(imgs).filterBounds(region).sort('system:time_start')

    # Regress VH ~ constant + t  → slope = coefficients[1,0]
    fit = col.select(['constant', 't', 'VH']).reduce(ee.Reducer.linearRegression(2, 1))
    slope = fit.select('coefficients').arrayGet([1, 0]).rename(f's1_vh_trend_{start_year}_{end_year}')
    return slope


def dw_annual_tree(year, prob_thresh, region):
    start = ee.Date.fromYMD(year,1,1); end = ee.Date.fromYMD(year,12,31)
    return (DW.filterBounds(region).filterDate(start, end)
              .select("trees").mean().gt(prob_thresh).rename(f"tree_{year}"))

def run_length_final(binary_images):
    # R_y = B_y * (1 + R_{y-1})
    init = ee.Image(0).toInt16()
    def step(b, acc):
        b = ee.Image(b); acc = ee.Image(acc)
        return b.multiply(acc.add(1)).toInt16()
    return ee.Image(ee.List(binary_images).iterate(step, init))

def chirps_sum(year, months, region):
    ims = []
    for m in months:
        start = ee.Date.fromYMD(year, m, 1); end = start.advance(1,"month")
        ims.append(CHIRPS.filterBounds(region).filterDate(start, end).sum())
    return ee.ImageCollection.fromImages(ims).sum()

def chirps_anom_jan_oct_2025(region):
    thisY = CHIRPS.filterBounds(region).filterDate("2025-01-01","2025-11-01").sum()
    clim = ee.ImageCollection.fromImages([
        CHIRPS.filterBounds(region).filterDate(ee.Date.fromYMD(y,1,1), ee.Date.fromYMD(y,11,1)).sum()
        for y in range(1981, 2025)
    ])
    mean = clim.mean(); std = clim.reduce(ee.Reducer.stdDev())
    return thisY.subtract(mean).divide(std).rename("chirps_anom_2025")

def burned_months_2025(region, buffer_m=5000):
    imgs = []
    for m in range(1, 13):
        start = ee.Date.fromYMD(2025, m, 1)
        end   = start.advance(1, "month")
        name  = f"burn_{m:02d}"

        # Slightly widen the query, still clip results back to AOI
        roi = region.buffer(buffer_m).bounds()

        coll = (MCD64
                .filterBounds(roi)
                .filterDate(start, end)
                .select("BurnDate"))

        # If the collection is empty, return a fully masked band for that month
        monthly = ee.Image(ee.Algorithms.If(
            coll.size().gt(0),
            coll.max().clip(region).gt(0).rename(name),
            _masked_const_band(name)  # from our earlier helpers
        ))
        imgs.append(monthly)

    # Sum 12 monthly booleans → number of burned months in 2025
    return ee.ImageCollection(imgs).sum().rename("mcd64_burn_months_2025")


def viirs_mean_2025(region):
    coll = VIIRSNTL.filterBounds(region).filterDate("2025-01-01","2025-11-01").select("avg_rad")
    return coll.mean().rename("viirs_ntl_mean_2025")

def viirs_trend_2019_2025(region):
    coll = (VIIRSNTL.filterBounds(region)
            .filterDate("2019-01-01","2025-11-01").select("avg_rad").sort("system:time_start"))
    def with_time(i):
        t = i.date().difference(ee.Date("2019-01-01"), "year")
        return i.addBands(ee.Image.constant(1).rename("constant")).addBands(ee.Image(t).rename("t"))
    fit = coll.map(with_time).select(["constant","t","avg_rad"]).reduce(ee.Reducer.linearRegression(2,1))
    return fit.select("coefficients").arrayGet([1,0]).rename("viirs_ntl_trend_2019_2025")

def h_spatial_percentiles(h_img, radius_m):
    # Compute local spatial percentiles of canopy height in a circular window.
    kernel = ee.Kernel.circle(radius=radius_m, units="meters", normalize=False)
    reducer = ee.Reducer.percentile([25, 50, 75, 98])
    p = h_img.reduceNeighborhood(
        reducer=reducer,
        kernel=kernel,
        skipMasked=True  # <-- pass as a keyword, not a positional boolean
        # You can also add: tileScale=2  # if you hit memory on large AOIs
    )
    return p.rename(["h_p25", "h_p50", "h_p75", "h_p98"])


In [7]:
# Cell — S2 helpers (mask + NDVI) for COPERNICUS/S2_SR_HARMONIZED

# Ensure the S2 collection handle exists
try:
    S2
except NameError:
    S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

def mask_s2(img):
    """
    Cloud/cirrus + shadow mask for Sentinel-2 SR (QA60 + SCL when available),
    scale reflectance to [0,1], and keep the key bands with friendly names.
    """
    # QA60 cloud bits: 10=clouds, 11=cirrus
    qa = img.select('QA60')
    cloudBitMask  = 1 << 10
    cirrusBitMask = 1 << 11
    qa_mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
              qa.bitwiseAnd(cirrusBitMask).eq(0))

    # SCL (scene classification) mask – skip cloud shadow(3), clouds(8), cirrus(10), snow(11)
    scl_mask = ee.Image(ee.Algorithms.If(
        img.bandNames().contains('SCL'),
        img.select('SCL').neq(3)
           .And(img.select('SCL').neq(8))
           .And(img.select('SCL').neq(10))
           .And(img.select('SCL').neq(11)),
        ee.Image(1)  # if SCL not present, do nothing
    ))

    mask = qa_mask.And(scl_mask)

    # Scale reflectance bands to [0,1] and rename for NDVI calc
    scaled = (img.select(['B2','B3','B4','B8'])
                .divide(10000.0)
                .rename(['blue','green','red','nir']))

    return (scaled.updateMask(mask)
                 .copyProperties(img, ['system:time_start']))

def add_ndvi(img):
    ndvi = img.normalizedDifference(['nir','red']).rename('NDVI')
    return img.addBands(ndvi)


In [8]:
# Cell 6 — Build features block-by-block (patched phenology section)

region = aoi  # alias

# A) Dynamic World run-length (strict & relaxed) to 2025
print("A) Computing Dynamic World run-length…")
years = list(range(2015, YEAR+1))
annual_strict = [ dw_annual_tree(y, 0.5, region) for y in years ]
annual_relax  = [ dw_annual_tree(y, 0.4, region) for y in years ]
a_strict = run_length_final(annual_strict).rename("a_years_continuous_2025")
a_relax  = run_length_final(annual_relax ).rename("a_years_continuous_relaxed_2025")
print("  ✓ Done A. Bands:", a_strict.addBands(a_relax).bandNames().getInfo())

# B) Height baseline + spatial percentiles
print("B) Computing height + neighborhood percentiles…")
h_mean = ee.Image(HEIGHT_ASSET).rename("h_mean").clip(region)
h_pcts = h_spatial_percentiles(h_mean, 90)
h_valid = h_mean.mask().reduce(ee.Reducer.anyNonZero()).rename("h_valid")
print("  ✓ Done B. Bands:", h_mean.addBands(h_pcts).addBands(h_valid).bandNames().getInfo())

# C) NDVI monthly (2025)
print("C) Computing S2 NDVI monthly medians for 2025…")
ndvi_images = [ s2_month_ndvi(YEAR, m, region) for m in range(1,13) ]
ndvi_stack = ee.ImageCollection.fromImages(ndvi_images).toBands()
print("  ✓ Done C. #bands:", len(ndvi_stack.bandNames().getInfo()))

# D) S1 seasonal stats + long-term trend
print("D) Computing S1 seasonal stats + trend…")
s1_dry    = s1_seasonal(YEAR, DRY_MONTHS, region).select(f"s1_vh_mean_{YEAR}")
s1_wetdif = s1_seasonal(YEAR, WET_MONTHS, region).select(f"s1_vv_vh_diff_{YEAR}")
s1_trend  = s1_vh_trend(2017, YEAR, region)
print("  ✓ Done D. Bands:", s1_dry.addBands(s1_wetdif).addBands(s1_trend).bandNames().getInfo())

# E) CHIRPS rainfall
print("E) Computing CHIRPS rainfall metrics…")
chirps_wet  = chirps_sum(YEAR, WET_MONTHS, region).rename("chirps_wet_sum_2025")
chirps_anom = chirps_anom_jan_oct_2025(region)
print("  ✓ Done E. Bands:", chirps_wet.addBands(chirps_anom).bandNames().getInfo())

# F) MODIS burned months (2025)
print("F) Counting burned months (MCD64)…")
burned_2025 = burned_months_2025(region)
print("  ✓ Done F. Bands:", burned_2025.bandNames().getInfo())

# G) VIIRS night-lights
print("G) Computing VIIRS NTL mean & trend…")
ntl_mean  = viirs_mean_2025(region)
ntl_trend = viirs_trend_2019_2025(region)
print("  ✓ Done G. Bands:", ntl_mean.addBands(ntl_trend).bandNames().getInfo())

# H) Phenology: coefficients → amplitude/phase (conditional)
print("H) Phenology (harmonics)…")
if 'PHENO_MAPPED' in globals() and PHENO_MAPPED is not None:
    phen = phen_derive(PHENO_MAPPED).clip(region)
    print("  ✓ Done H. Bands:", phen.bandNames().getInfo())
else:
    print("  ↷ Phenology asset not available — using masked placeholders to keep schema stable.")
    # Create masked placeholders so band schema stays identical even without the asset
    phen = (ee.Image.constant([0,0,0,0,0,0,0,0])
            .rename(['phen_b0','phen_b1','phen_b2','phen_b3','phen_b4',
                     'phen_A1','phen_phi1','phen_A2','phen_phi2'][:8])  # 8 bands total
            .updateMask(ee.Image(0)))
    print("  ✓ Placeholder phenology bands added:", phen.bandNames().getInfo())


A) Computing Dynamic World run-length…
  ✓ Done A. Bands: ['a_years_continuous_2025', 'a_years_continuous_relaxed_2025']
B) Computing height + neighborhood percentiles…
  ✓ Done B. Bands: ['h_mean', 'h_p25', 'h_p50', 'h_p75', 'h_p98', 'h_valid']
C) Computing S2 NDVI monthly medians for 2025…
  ✓ Done C. #bands: 12
D) Computing S1 seasonal stats + trend…
  ✓ Done D. Bands: ['s1_vh_mean_2025', 's1_vv_vh_diff_2025', 's1_vh_trend_2017_2025']
E) Computing CHIRPS rainfall metrics…
  ✓ Done E. Bands: ['chirps_wet_sum_2025', 'chirps_anom_2025']
F) Counting burned months (MCD64)…
  ✓ Done F. Bands: ['mcd64_burn_months_2025']
G) Computing VIIRS NTL mean & trend…
  ✓ Done G. Bands: ['viirs_ntl_mean_2025', 'viirs_ntl_trend_2019_2025']
H) Phenology (harmonics)…
  ↷ Phenology asset not available — using masked placeholders to keep schema stable.
  ✓ Placeholder phenology bands added: ['phen_b0', 'phen_b1', 'phen_b2', 'phen_b3', 'phen_b4', 'phen_A1', 'phen_phi1', 'phen_A2']


In [9]:

feature_stack = (
    a_strict.addBands(a_relax)
            .addBands(h_mean).addBands(h_pcts).addBands(h_valid)
            .addBands(ndvi_stack)
            .addBands(s1_dry).addBands(s1_wetdif).addBands(s1_trend)
            .addBands(chirps_wet).addBands(chirps_anom)
            .addBands(burned_2025)
            .addBands(ntl_mean).addBands(ntl_trend)
)

if phen is not None:
    feature_stack = feature_stack.addBands(phen)
    print("✓ Phenology bands present in stack.")
else:
    print("↷ Phenology bands not added (None).")

feature_stack = feature_stack.toFloat().reproject(CRS, None, SCALE).clip(region)

bands = feature_stack.bandNames().getInfo()
print("Final feature stack bands (", len(bands), "):")
for i, b in enumerate(bands, 1):
    print(f"  {i:02d}. {b}")


✓ Phenology bands present in stack.
Final feature stack bands ( 36 ):
  01. a_years_continuous_2025
  02. a_years_continuous_relaxed_2025
  03. h_mean
  04. h_p25
  05. h_p50
  06. h_p75
  07. h_p98
  08. h_valid
  09. 0_ndvi_2025_01
  10. 1_ndvi_2025_02
  11. 2_ndvi_2025_03
  12. 3_ndvi_2025_04
  13. 4_ndvi_2025_05
  14. 5_ndvi_2025_06
  15. 6_ndvi_2025_07
  16. 7_ndvi_2025_08
  17. 8_ndvi_2025_09
  18. 9_ndvi_2025_10
  19. 10_ndvi_2025_11
  20. 11_ndvi_2025_12
  21. s1_vh_mean_2025
  22. s1_vv_vh_diff_2025
  23. s1_vh_trend_2017_2025
  24. chirps_wet_sum_2025
  25. chirps_anom_2025
  26. mcd64_burn_months_2025
  27. viirs_ntl_mean_2025
  28. viirs_ntl_trend_2019_2025
  29. phen_b0
  30. phen_b1
  31. phen_b2
  32. phen_b3
  33. phen_b4
  34. phen_A1
  35. phen_phi1
  36. phen_A2


In [10]:
# ONE CELL — sample ONE pixel and print ALL feature values one-by-one (safe trend + safe burned)

import ee, math

# -------- expects these from your setup --------
# aoi, YEAR, SCALE, CRS
# a_strict, a_relax, h_mean, h_pcts, h_valid, ndvi_stack,
# s1_dry, s1_wetdif, chirps_wet, chirps_anom,
# ntl_mean  (we will recompute a safe ntl_trend),
# phen (optional; can be placeholder or real ee.Image)

need = ['aoi','YEAR','SCALE','CRS',
        'a_strict','a_relax','h_mean','h_pcts','h_valid',
        'ndvi_stack','s1_dry','s1_wetdif',
        'chirps_wet','chirps_anom','ntl_mean']
missing = [v for v in need if v not in globals()]
if missing:
    raise RuntimeError(f"Missing: {missing}. Run your previous cells first.")

# ---------- helpers (safe) ----------
def _masked_const_band(name, value=0):
    return ee.Image.constant(value).rename(name).updateMask(ee.Image(0))

# Burned months (homogeneous)
def burned_months_2025_fixed(region, buffer_m=5000):
    MCD64 = ee.ImageCollection("MODIS/061/MCD64A1")
    ims = []
    for m in range(1, 13):
        start = ee.Date.fromYMD(2025, m, 1); end = start.advance(1, "month")
        coll  = MCD64.filterBounds(region.buffer(buffer_m).bounds()).filterDate(start, end).select("BurnDate")
        monthly = ee.Image(ee.Algorithms.If(
            coll.size().gt(0),
            coll.max().clip(region).gt(0).rename("burn_flag").toByte(),
            _masked_const_band("burn_flag")
        ))
        ims.append(monthly)
    return ee.ImageCollection.fromImages(ims).sum().rename("mcd64_burn_months_2025").toInt16()

# S1 simple VH trend (no 't' band)
S1 = ee.ImageCollection("COPERNICUS/S1_GRD")
def s1_month_safe(year, month, region, buffer_m=2000):
    start = ee.Date.fromYMD(year, month, 1); end = start.advance(1, "month")
    roi = region.buffer(buffer_m).bounds()
    coll = (S1.filterBounds(roi).filterDate(start, end)
             .filter(ee.Filter.eq("instrumentMode","IW"))
             .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VH"))
             .select(["VH"]))
    return ee.Image(ee.Algorithms.If(coll.size().gt(0), coll.median().clip(region), _masked_const_band("VH")))

def s1_vh_trend_simple(region, start_year=2017, end_year=YEAR):
    imgs = []
    for y in range(start_year, end_year+1):
        for m in range(1,13):
            imgs.append(s1_month_safe(y, m, region).set("system:time_start", ee.Date.fromYMD(y,m,15).millis()))
    ts = ee.ImageCollection(imgs).filterBounds(region).sort("system:time_start")
    first = ee.Image(ts.first()); last = ee.Image(ts.sort("system:time_start", False).first())
    yrs = ee.Number(ee.Date(last.get("system:time_start")).difference(ee.Date(first.get("system:time_start")), "year")).max(1/12)
    return last.subtract(first).divide(yrs).rename(f"s1_vh_trend_2017_{end_year}")

VIIRS = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
def viirs_trend_simple(region, start="2019-01-01", end="2025-11-01"):
    col = VIIRS.filterBounds(region).filterDate(start, end).select("avg_rad").sort("system:time_start")
    first = ee.Image(col.first()); last = ee.Image(col.sort("system:time_start", False).first())
    yrs = ee.Number(ee.Date(last.get("system:time_start")).difference(ee.Date(first.get("system:time_start")), "year")).max(1/12)
    return last.subtract(first).divide(yrs).rename("viirs_ntl_trend_2019_2025")

burn_fixed   = burned_months_2025_fixed(aoi)
s1_trend_safe= s1_vh_trend_simple(aoi, 2017, YEAR)
ntl_trend_safe = viirs_trend_simple(aoi)

img = (a_strict.addBands(a_relax)
              .addBands(h_mean).addBands(h_pcts).addBands(h_valid)
              .addBands(ndvi_stack)
              .addBands(s1_dry).addBands(s1_wetdif).addBands(s1_trend_safe)
              .addBands(chirps_wet).addBands(chirps_anom)
              .addBands(burn_fixed)
              .addBands(ntl_mean).addBands(ntl_trend_safe))

if 'phen' in globals() and isinstance(phen, ee.image.Image):
    img = img.addBands(phen)

PIXEL_LON = None   # e.g., 77.175
PIXEL_LAT = None   # e.g., 28.532
pt = (ee.Geometry.Point([float(PIXEL_LON), float(PIXEL_LAT)]) if PIXEL_LON is not None and PIXEL_LAT is not None
      else ee.Geometry(aoi).centroid(SCALE))

# ---------- read all band values at that pixel ----------
d = img.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=pt,
        scale=SCALE,
        bestEffort=True,
        maxPixels=1e9
    )

vals = d.getInfo() or {}

lonlat = pt.coordinates().getInfo()
print(f"Pixel @ lon,lat = {lonlat[0]:.6f}, {lonlat[1]:.6f}\n")

bands = img.bandNames().getInfo()
for i, b in enumerate(bands, 1):
    v = vals.get(b, None)
    if isinstance(v, list) and len(v)==1: v = v[0]
    if isinstance(v, (int,float)):
        if not math.isfinite(float(v)): v = None
    out = "NA" if v is None else (f"{float(v):.6g}" if isinstance(v,(int,float)) else str(v))
    print(f"{i:02d}. {b} = {out}")


Pixel @ lon,lat = 77.175000, 28.530000

01. a_years_continuous_2025 = 3
02. a_years_continuous_relaxed_2025 = 6
03. h_mean = 12
04. h_p25 = 11
05. h_p50 = 12
06. h_p75 = 13
07. h_p98 = 15
08. h_valid = 1
09. 0_ndvi_2025_01 = 0.694313
10. 1_ndvi_2025_02 = 0.58837
11. 2_ndvi_2025_03 = 0.708138
12. 3_ndvi_2025_04 = 0.561471
13. 4_ndvi_2025_05 = 0.663921
14. 5_ndvi_2025_06 = 0.704268
15. 6_ndvi_2025_07 = NA
16. 7_ndvi_2025_08 = 0.857962
17. 8_ndvi_2025_09 = 0.769898
18. 9_ndvi_2025_10 = 0.826731
19. 10_ndvi_2025_11 = NA
20. 11_ndvi_2025_12 = NA
21. s1_vh_mean_2025 = -14.1617
22. s1_vv_vh_diff_2025 = 6.04647
23. s1_vh_trend_2017_2025 = NA
24. chirps_wet_sum_2025 = 710.057
25. chirps_anom_2025 = 1.11361
26. mcd64_burn_months_2025 = NA
27. viirs_ntl_mean_2025 = 15.55
28. viirs_ntl_trend_2019_2025 = 0.751254
29. phen_b0 = NA
30. phen_b1 = NA
31. phen_b2 = NA
32. phen_b3 = NA
33. phen_b4 = NA
34. phen_A1 = NA
35. phen_phi1 = NA
36. phen_A2 = NA
